<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/feature%2Frefactor-colab/colabs/cross_app_access_demo_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuration Variables

Set all required configuration values below. These will be used throughout the notebook.

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values for your Okta environment
# ============================================================================

# Okta Domain
OKTA_DOMAIN = "https://{YOUR_DOMAIN}.oktapreview.com"  # Your Okta domain - no trailing slash

# OAuth 2.0 Application Settings
CLIENT_ID = ""           # OAuth 2.0 Client ID
CLIENT_SECRET = ""   # OAuth 2.0 Client Secret (required)
REDIRECT_URI = "http://localhost:8080/authorization-code/callback"  # Registered redirect URI

# Custom Authorization Server for your resource/MCP server
AUTHORIZATION_SERVER_ID = ""

# Agent / Workload Principal Configuration (for JWT bearer assertion)
PRINCIPAL_ID = ""     # Agent identifier

# Private JWK for JWT bearer assertion
# Replace with your actual agent private JWK (RSA key)
PRIVATE_JWK = {}


# Resource/MCP Server Audience (for token verification in Step 4)
RESOURCE_SERVER_AUDIENCE = ""  # Your resource server audience

# ID Token (will be obtained in STEP 0, or provide your own)
ID_TOKEN = None  # Leave as None to obtain via authorization code flow

# ============================================================================
from google.colab import userdata

print("Configuration loaded successfully!")
print(f"Okta Domain: {OKTA_DOMAIN}")
print(f"Client ID: {CLIENT_ID[:20]}..." if len(CLIENT_ID) > 20 else f"Client ID: {CLIENT_ID}")
print(f"Authorization Server: {AUTHORIZATION_SERVER_ID}")
print(f"Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")

Configuration loaded successfully!
Okta Domain: https://atko-ai.oktapreview.com
Client ID: 0oauain3t7W5RDPiy1d7
Authorization Server: ausubr9dq7o80RBml1d7
Resource Server Audience: https://benefits.streamward.com


# Okta Cross-App Access Demo

This notebook demonstrates the complete **Identity Assertion Authorization Grant (ID-JAG)** flow for secure cross-application access using the Okta AI SDK.


## The 4-Step Cross App Access Flow

1. **Exchange ID Token for ID-JAG Token** - Convert user's ID token to cross-app token
2. **Verify ID-JAG Token** - Validate the token's authenticity and claims
3. **Exchange ID-JAG for Auth Server Token** - Get access token for target application
4. **Verify Auth Server Token** - Validate final access token before use

---

## Setup and Installation

First, install the SDK and configure all required variables.

In [ ]:
# Install the Okta AI SDK from PyPI
!pip install -q okta-ai-sdk-proto

In [ ]:
# Import required modules
import json
import os
from datetime import datetime
from okta_ai_sdk import OktaAISDK, OktaAIConfig
from okta_ai_sdk.types import AuthServerTokenRequest

print("Imports successful!")

Imports successful!


## Configuration

### Prerequisites

Before running this notebook, ensure you have followed the PoC guide through use case **2. Secure** to configure this lab.

---

## STEP 0: Obtain ID Token (Optional)

If you don't have an ID token yet, use this section to obtain one through OAuth 2.0 authorization code flow.

### Important:
This step uses the **Org Authorization Server** (_not a custom authorization server_). The ID token will be issued directly by your Okta domain:
- Issuer: `https://your-domain.okta.com`
- Endpoint: `/oauth2/v1/authorize` and `/oauth2/v1/token`

This is the correct approach for obtaining the initial user ID token that will be used in the ID-JAG flow.

### The Flow:

1. **Generate state and nonce** - For CSRF protection and ID token validation
2. **Build authorization URL** - User authenticates in browser (Org Authorization Server)
3. **Copy authorization code** - From the redirect URL after authentication
4. **Exchange code for tokens** - Get ID token with issuer = Okta domain

In [ ]:
# OAuth 2.0 Configuration for obtaining ID token
import base64
import hashlib
import secrets
import urllib.parse
from IPython.display import display, HTML
import requests

# IMPORTANT: Initial ID token generation MUST use Org Authorization Server
# The issuer will be the Okta domain directly (e.g., "https://your-domain.okta.com")
# NOT a custom authorization server

print("OAuth 2.0 Configuration for Initial ID Token")
print("=" * 60)
print(f"Okta Domain: {OKTA_DOMAIN}")
print(f"Client ID: {CLIENT_ID}")
print(f"Redirect URI: {REDIRECT_URI}")
print(f"Authorization Server: Org Authorization Server (default)")
print(f"Token Issuer: {OKTA_DOMAIN}")
print("=" * 60)

OAuth 2.0 Configuration for Initial ID Token
Okta Domain: https://atko-ai.oktapreview.com
Client ID: 0oauain3t7W5RDPiy1d7
Redirect URI: http://localhost:8080/authorization-code/callback
Authorization Server: Org Authorization Server (default)
Token Issuer: https://atko-ai.oktapreview.com


### Generate State and Nonce

Generate security parameters for the authorization flow.

In [ ]:
# Generate state and nonce for authorization flow
state = secrets.token_urlsafe(32)
nonce = secrets.token_urlsafe(32)

print("Security Parameters Generated:")
print(f"   State: {state[:30]}...")
print(f"   Nonce: {nonce[:30]}...")
print("\nNote: These values protect against CSRF attacks and ensure ID token validity")

Security Parameters Generated:
   State: _i51lwHv5LNroq5v5zbSEMJQZLV8YX...
   Nonce: zxyktBIU5OG-dV0cdxw90Im7ALLdZ4...

Note: These values protect against CSRF attacks and ensure ID token validity


### Build and Open Authorization URL

Click the button below to open the authorization URL in a new browser tab.

In [ ]:
# Build authorization URL using Org Authorization Server
def build_authorization_url(
    okta_domain: str,
    client_id: str,
    redirect_uri: str,
    state: str,
    nonce: str,
    scopes: list = None
) -> str:
    """
    Build OAuth 2.0 authorization URL for Org Authorization Server

    Args:
        okta_domain: Your Okta domain
        client_id: OAuth client ID
        redirect_uri: Registered redirect URI
        state: State parameter for CSRF protection
        nonce: Nonce for ID token validation
        scopes: List of OAuth scopes (default: ['openid', 'profile', 'email'])

    Returns:
        Complete authorization URL for Org Authorization Server

    Note:
        This ALWAYS uses the Org Authorization Server (/oauth2/v1/authorize)
        The ID token issuer will be the Okta domain directly
    """
    if scopes is None:
        scopes = ['openid', 'profile', 'email']

    # Always use Org Authorization Server for initial ID token
    auth_endpoint = f"{okta_domain}/oauth2/v1/authorize"

    # Build query parameters
    params = {
        'client_id': client_id,
        'response_type': 'code',
        'scope': ' '.join(scopes),
        'redirect_uri': redirect_uri,
        'state': state,
        'nonce': nonce
    }

    # Build full URL
    authorization_url = f"{auth_endpoint}?{urllib.parse.urlencode(params)}"
    return authorization_url

# Build the authorization URL using Org Authorization Server
authorization_url = build_authorization_url(
    okta_domain=OKTA_DOMAIN,
    client_id=CLIENT_ID,
    redirect_uri=REDIRECT_URI,
    state=state,
    nonce=nonce,
    scopes=['openid', 'profile', 'email']  # Customize scopes as needed
)

print("Authorization URL generated!")
print(f"\n{authorization_url}")
print("\n" + "="*80)

# Display clickable button
html_button = f"""
<div style="margin: 20px 0;">
    <a href="{authorization_url}" target="_blank" style="
        display: inline-block;
        padding: 15px 30px;
        background-color: #007bff;
        color: white;
        text-decoration: none;
        border-radius: 5px;
        font-weight: bold;
        font-size: 16px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    ">Click Here to Authenticate with Okta</a>
</div>
<div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
    <strong>Instructions:</strong>
    <ol style="margin: 10px 0 0 0;">
        <li>Click the button above to open the authorization URL in a new tab</li>
        <li>Sign in with your Okta credentials</li>
        <li>After authentication, you'll be redirected to: <code>{REDIRECT_URI}</code></li>
        <li>Copy the <strong>code</strong> parameter from the URL (it will look like: <code>?code=ABC123...</code>)</li>
        <li>Paste the code in the next cell to exchange it for tokens</li>
    </ol>
</div>
"""

display(HTML(html_button))

print("\nWhat to do next:")
print("   1. Click the button above")
print("   2. Sign in to Okta")
print("   3. Copy the 'code' parameter from the redirect URL")
print("   4. Run the next script -- it will prompt you for the code from this step.")
print("\nNote: The ID token will be issued by the Org Authorization Server")
print(f"      Issuer: {OKTA_DOMAIN}")

Authorization URL generated!

https://atko-ai.oktapreview.com/oauth2/v1/authorize?client_id=0oauain3t7W5RDPiy1d7&response_type=code&scope=openid+profile+email&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2Fauthorization-code%2Fcallback&state=_i51lwHv5LNroq5v5zbSEMJQZLV8YX0liKmd3IIJR7A&nonce=zxyktBIU5OG-dV0cdxw90Im7ALLdZ41hjiGrUIoEbfc




What to do next:
   1. Click the button above
   2. Sign in to Okta
   3. Copy the 'code' parameter from the redirect URL
   4. Paste it in the next cell

Note: The ID token will be issued by the Org Authorization Server
      Issuer: https://atko-ai.oktapreview.com


### Exchange Authorization Code for Tokens

Run the following script. You will be prompted to enter the authorization code you copied from the previous script. Paste the code and submit (_Enter_).

In [ ]:
# Exchange authorization code for tokens using Org Authorization Server
def exchange_code_for_tokens(
    okta_domain: str,
    client_id: str,
    client_secret: str,
    redirect_uri: str,
    code: str
) -> dict:
    """
    Exchange authorization code for tokens using Org Authorization Server

    Args:
        okta_domain: Your Okta domain
        client_id: OAuth client ID
        client_secret: OAuth client secret
        redirect_uri: Registered redirect URI (must match authorization request)
        code: Authorization code from redirect URL

    Returns:
        Dictionary containing tokens and metadata

    Note:
        This ALWAYS uses the Org Authorization Server (/oauth2/v1/token)
        The ID token issuer will be the Okta domain directly
    """
    # Always use Org Authorization Server for initial ID token
    token_endpoint = f"{okta_domain}/oauth2/v1/token"

    # Prepare token request
    data = {
        'grant_type': 'authorization_code',
        'client_id': client_id,
        'client_secret': client_secret,
        'redirect_uri': redirect_uri,
        'code': code
    }

    # Make token request
    response = requests.post(
        token_endpoint,
        data=data,
        headers={'Content-Type': 'application/x-www-form-urlencoded'}
    )

    # Check response
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Token exchange failed: {response.status_code} - {response.text}")

# Paste your authorization code here (from the redirect URL)
# Example: If redirect URL is https://example.com?code=ABC123&state=xyz
# Then paste just the code value: ABC123

AUTHORIZATION_CODE = input("\nPaste the authorization code here: ").strip()

if not AUTHORIZATION_CODE:
    print("\nNo code provided. Please paste the authorization code and run this cell again.")
else:
    print("\nExchanging authorization code for tokens...\n")

    try:
        # Exchange code for tokens using Org Authorization Server
        token_response = exchange_code_for_tokens(
            okta_domain=OKTA_DOMAIN,
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
            redirect_uri=REDIRECT_URI,
            code=AUTHORIZATION_CODE
        )

        print("Token exchange successful!\n")
        print("Token Response:")
        print(f"   Token Type: {token_response.get('token_type')}")
        print(f"   Expires In: {token_response.get('expires_in')} seconds")
        print(f"   Scope: {token_response.get('scope', 'N/A')}")

        # Extract tokens
        ID_TOKEN = token_response.get('id_token')
        ACCESS_TOKEN = token_response.get('access_token')
        REFRESH_TOKEN = token_response.get('refresh_token')

        print(f"\nTokens Obtained:")
        if ID_TOKEN:
            print(f"   [OK] ID Token: {ID_TOKEN[:50]}...")
        if ACCESS_TOKEN:
            print(f"   [OK] Access Token: {ACCESS_TOKEN[:50]}...")
        if REFRESH_TOKEN:
            print(f"   [OK] Refresh Token: {REFRESH_TOKEN[:50]}...")

        # Decode and display ID token claims (optional)
        if ID_TOKEN:
            import jwt
            decoded = jwt.decode(ID_TOKEN, options={"verify_signature": False})
            print(f"\nID Token Claims:")
            print(f"   Subject: {decoded.get('sub')}")
            print(f"   Email: {decoded.get('email', 'N/A')}")
            print(f"   Name: {decoded.get('name', 'N/A')}")
            print(f"   Issuer: {decoded.get('iss')}")
            print(f"   Audience: {decoded.get('aud')}")

            # Verify issuer is the Okta domain (Org Authorization Server)
            if decoded.get('iss') == OKTA_DOMAIN:
                print(f"\n   [OK] Token issued by Org Authorization Server: {OKTA_DOMAIN}")
            else:
                print(f"\n   [WARNING] Unexpected issuer: {decoded.get('iss')}")
                print(f"   Expected: {OKTA_DOMAIN}")

        print("\n" + "="*80)
        print("You can now use the ID_TOKEN variable in the cross-app access flow below!")
        print("="*80)

    except Exception as e:
        print(f"Error during token exchange: {e}")
        print("\nTroubleshooting:")
        print("   • Make sure you copied the entire authorization code")
        print("   • Verify your redirect_uri matches what's registered in Okta")
        print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)")
        print("   • Ensure your client_id and client_secret are correct")


Paste the authorization code here: BTPKG-2wS0IYVnUrIEQkG9kNQrCh48W7LdVvBVALstA

Exchanging authorization code for tokens...

Token exchange successful!

Token Response:
   Token Type: Bearer
   Expires In: 3600 seconds
   Scope: openid profile email

Tokens Obtained:
   [OK] ID Token: eyJraWQiOiJtSUM2R0ZkMUw1OGUwMHdyT1pqODlSWEZybnJiZV...
   [OK] Access Token: eyJraWQiOiJtSUM2R0ZkMUw1OGUwMHdyT1pqODlSWEZybnJiZV...

ID Token Claims:
   Subject: 00utndie9krbalhn51d7
   Email: danny.fuhriman@okta.com
   Name: Danny Fuhriman
   Issuer: https://atko-ai.oktapreview.com
   Audience: 0oauain3t7W5RDPiy1d7

   [OK] Token issued by Org Authorization Server: https://atko-ai.oktapreview.com

You can now use the ID_TOKEN variable in the cross-app access flow below!


---

### Check Point!
Run the following script to ensure you have completed the necessary steps to continue.

In [ ]:
import os
import json
from getpass import getpass

# Verify configuration is set
print("Cross-App Access Configuration")
print("=" * 60)

if ID_TOKEN and ID_TOKEN != "your_id_token_here" and ID_TOKEN is not None:
    print(f"[OK] Using ID Token from STEP 0: {ID_TOKEN[:30]}...")
else:
    print("[WARNING] No ID token set. Please run STEP 0 or set ID_TOKEN manually.")
    print("          The ID-JAG flow will fail without a valid ID token.")

print(f"\nConfiguration Summary:")
print(f"   Okta Domain: {OKTA_DOMAIN}")
print(f"   Client ID: {CLIENT_ID[:20]}..." if len(CLIENT_ID) > 20 else f"   Client ID: {CLIENT_ID}")
print(f"   Auth Server: {AUTHORIZATION_SERVER_ID}")
print(f"   Principal ID: {PRINCIPAL_ID[:20]}..." if len(PRINCIPAL_ID) > 20 else f"   Principal ID: {PRINCIPAL_ID}")
print(f"   Private JWK: {'Configured' if PRIVATE_JWK.get('kid') != 'your_key_id' else 'Placeholder'}")
print(f"   Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")
print("=" * 60)

Cross-App Access Configuration
[OK] Using ID Token from STEP 0: eyJraWQiOiJtSUM2R0ZkMUw1OGUwMH...

Configuration Summary:
   Okta Domain: https://atko-ai.oktapreview.com
   Client ID: 0oauain3t7W5RDPiy1d7
   Auth Server: ausubr9dq7o80RBml1d7
   Principal ID: wlpubo1xjrf1B1NRO1d7
   Private JWK: Configured
   Resource Server Audience: https://benefits.streamward.com


### Initialize the Okta AI SDK

Create an SDK instance with cross-app access configuration.

In [ ]:
# Initialize SDK with JWT bearer assertion support
config = OktaAIConfig(
    okta_domain=OKTA_DOMAIN,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    authorization_server_id=AUTHORIZATION_SERVER_ID,
    principal_id=PRINCIPAL_ID,
    private_jwk=PRIVATE_JWK
)

sdk = OktaAISDK(config)

print("SDK initialized successfully!")
print(f"   Cross-App Access Client: {sdk.cross_app_access}")

SDK initialized successfully!
   Cross-App Access Client: <okta_ai_sdk.cross_app_access.client.CrossAppAccessClient object at 0x7f656ce33680>


---

## Step 1: Exchange ID Token for ID-JAG Token

The first step is to exchange a user's Okta ID token for an ID-JAG token. This token represents the user's identity and can be used for cross-application access.

### What happens:
1. SDK generates a JWT bearer assertion using your private key
1. Calls Okta's org authorization server token endpoint
1. Exchanges ID token for ID-JAG token with specified audience
1. Returns ID-JAG token with metadata

In [ ]:
# Define the audience for the ID-JAG token
# This should match your authorization server
id_jag_audience = f"{OKTA_DOMAIN}/oauth2/{AUTHORIZATION_SERVER_ID}"

print(f"Step 1: Exchanging ID token for ID-JAG token...")
print(f"   Audience: {id_jag_audience}")
print(f"   Requested Scope: mcp:read\n")


try:
    # Exchange ID token for ID-JAG token
    id_jag_result = sdk.cross_app_access.exchange_token(
        token=ID_TOKEN,
        token_type="id_token",
        audience=id_jag_audience,
        scope="mcp:read"  # Optional: request specific scopes
    )

    print("[SUCCESS] ID-JAG token obtained successfully!")
    print(f"\nToken Details:")
    print(f"   Token Type: {id_jag_result.token_type}")
    print(f"   Issued Token Type: {id_jag_result.issued_token_type}")
    print(f"   Expires In: {id_jag_result.expires_in} seconds")
    print(f"   Scope: {id_jag_result.scope or 'N/A'}")
    print(f"   Token (first 50 chars): {id_jag_result.access_token[:50]}...")

    # Store for next step
    id_jag_token = id_jag_result.access_token

except Exception as e:
    print(f"[ERROR] Error: {e}")
    raise

Step 1: Exchanging ID token for ID-JAG token...
   Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
   Requested Scope: mcp:read

 Exchanging ID token for ID-JAG token...
 Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
 Using JWT bearer assertion for client authentication
 Making ID-JAG token exchange request to: https://atko-ai.oktapreview.com/oauth2/v1/token
 ID-JAG token exchange successful
 Issued token type: urn:ietf:params:oauth:token-type:id-jag
 Expires in: 300 seconds
[SUCCESS] ID-JAG token obtained successfully!

Token Details:
   Token Type: N_A
   Issued Token Type: urn:ietf:params:oauth:token-type:id-jag
   Expires In: 300 seconds
   Scope: N/A
   Token (first 50 chars): eyJraWQiOiJtSUM2R0ZkMUw1OGUwMHdyT1pqODlSWEZybnJiZV...


---

## Step 2: Verify ID-JAG Token

Before using the ID-JAG token, we should verify its authenticity. This step:
1. Validates the token signature using Okta's public keys (JWKS)
1. Checks the audience, issuer, and expiration claims
1. Extracts user information from the token

### Security Note:
_Always verify tokens before trusting their contents._

This prevents:
- Tampered tokens
- Expired tokens
- Tokens meant for different audiences

In [ ]:
print(f"Step 2: Verifying ID-JAG token...")
print(f"   Expected Audience: {id_jag_audience}\n")

try:
    # Verify the ID-JAG token
    verification_result = sdk.cross_app_access.verify_id_jag_token(
        token=id_jag_token,
        audience=id_jag_audience
    )

    if verification_result.valid:
        print("[SUCCESS] ID-JAG token is VALID!")
        print(f"\nToken Claims:")
        print(f"   Subject: {verification_result.sub}")
        print(f"   Email: {verification_result.email or 'N/A'}")
        print(f"   Issuer: {verification_result.iss}")
        print(f"   Audience: {verification_result.aud}")

        # Format expiration time
        if verification_result.exp:
            exp_time = datetime.fromtimestamp(verification_result.exp)
            print(f"   Expires At: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")

        # Show full payload (optional)
        print(f"\nFull Payload:")
        print(json.dumps(verification_result.payload, indent=2))
    else:
        print(f"[ERROR] Token verification FAILED!")
        print(f"   Error: {verification_result.error}")
        raise Exception(f"Token verification failed: {verification_result.error}")

except Exception as e:
    print(f"[ERROR] Error during verification: {e}")
    raise

Step 2: Verifying ID-JAG token...
   Expected Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7

 Verifying ID-JAG token...
 Expected Issuer: https://atko-ai.oktapreview.com
 Expected Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
 JWKS URI: https://atko-ai.oktapreview.com/oauth2/v1/keys
 ID-JAG token verified successfully
 Subject: 00utndie9krbalhn51d7
 Email: N/A
 Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
 Issuer: https://atko-ai.oktapreview.com
 Expires: 2026-02-03 05:34:59
[SUCCESS] ID-JAG token is VALID!

Token Claims:
   Subject: 00utndie9krbalhn51d7
   Email: N/A
   Issuer: https://atko-ai.oktapreview.com
   Audience: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
   Expires At: 2026-02-03 05:34:59

Full Payload:
{
  "jti": "IDAAG.17sW7ioD44d3ltQxyMveA1CWBCwqo08JHYwAk_UcHhQ",
  "iss": "https://atko-ai.oktapreview.com",
  "aud": "https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7",
  "iat

---

## Step 3: Exchange ID-JAG Token for Authorization Server Token

Now we exchange the verified ID-JAG token for an access token from a custom authorization server. This token can be used to access protected resources.

### What happens:
1. SDK generates a JWT bearer assertion for the authorization server
1. Uses the ID-JAG token as the assertion
1. Calls the custom authorization server's token endpoint
1. Returns an access token with appropriate scopes

### Use Case:
This is useful when you need to access resources protected by a custom authorization server with specific scopes and policies.

In [ ]:
print(f"Step 3: Exchanging ID-JAG token for authorization server token...")
print(f"   Authorization Server: {AUTHORIZATION_SERVER_ID}\n")

try:
    # Create request for authorization server token
    auth_server_request = AuthServerTokenRequest(
        id_jag_token=id_jag_token,
        authorization_server_id=AUTHORIZATION_SERVER_ID,
        principal_id=PRINCIPAL_ID,
        private_jwk=PRIVATE_JWK
    )

    # Exchange for authorization server token
    auth_server_result = sdk.cross_app_access.exchange_id_jag_for_auth_server_token(
        auth_server_request
    )

    print("[SUCCESS] Authorization server token obtained successfully!")
    print(f"\nToken Details:")
    print(f"   Token Type: {auth_server_result.token_type}")
    print(f"   Expires In: {auth_server_result.expires_in} seconds")
    print(f"   Scope: {auth_server_result.scope or 'N/A'}")
    print(f"   Token (first 50 chars): {auth_server_result.access_token[:50]}...")

    if auth_server_result.refresh_token:
        print(f"   Refresh Token: Available")

    # Store for next step
    auth_server_token = auth_server_result.access_token

except Exception as e:
    print(f"[ERROR] Error: {e}")
    raise

Step 3: Exchanging ID-JAG token for authorization server token...
   Authorization Server: ausubr9dq7o80RBml1d7

 Exchanging ID-JAG token for authorization server token...
 Authorization Server ID: ausubr9dq7o80RBml1d7
 Using JWT bearer assertion for client authentication
 Making authorization server token request to: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7/v1/token
 Authorization server token exchange successful
 Expires in: 3600 seconds
 Scope: mcp:read
[SUCCESS] Authorization server token obtained successfully!

Token Details:
   Token Type: Bearer
   Expires In: 3600 seconds
   Scope: mcp:read
   Token (first 50 chars): eyJraWQiOiJtN0RZQnpMVGRIM2VKQkgzU2ZDbVM5MmU0dVhORG...


---

## Step 4: Verify Authorization Server Token

The final step is to verify the authorization server token before using it to access resources.

### What happens:
1. Validates the token signature using the authorization server's public keys
1. Checks audience, issuer, and expiration
1. Extracts scope and user information

### Important:
The resource server (your API) should perform this verification on every request to ensure:
- Token is valid and not tampered with
- Token is not expired
- Token is intended for this resource server (audience check)
- User has required permissions (scope check)

In [ ]:
# Use the resource server audience from configuration
print(f"Step 4: Verifying authorization server token...")
print(f"   Expected Audience: {RESOURCE_SERVER_AUDIENCE}\n")

try:
    # Verify the authorization server token
    verification_result = sdk.cross_app_access.verify_auth_server_token(
        token=auth_server_token,
        authorization_server_id=AUTHORIZATION_SERVER_ID,
        audience=RESOURCE_SERVER_AUDIENCE
    )

    if verification_result.valid:
        print("[SUCCESS] Authorization server token is VALID!")
        print(f"\nToken Claims:")
        print(f"   Subject: {verification_result.sub}")
        print(f"   Email: {verification_result.email or 'N/A'}")
        print(f"   Issuer: {verification_result.iss}")
        print(f"   Audience: {verification_result.aud}")
        print(f"   Scope: {verification_result.scope or 'N/A'}")

        # Format expiration time
        if verification_result.exp:
            exp_time = datetime.fromtimestamp(verification_result.exp)
            print(f"   Expires At: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")

        # Show full payload (optional)
        print(f"\nFull Payload:")
        print(json.dumps(verification_result.payload, indent=2))

        # Check for specific scopes (example)
        if verification_result.scope:
            scopes = verification_result.scope.split()
            print(f"\nGranted Scopes: {', '.join(scopes)}")

            # Example: Check for required scope
            if 'mcp:read' in scopes:
                print("   [OK] Has 'mcp:read' permission")
            else:
                print("   [WARNING] Missing 'mcp:read' permission")
    else:
        print(f"[ERROR] Token verification FAILED!")
        print(f"   Error: {verification_result.error}")
        raise Exception(f"Token verification failed: {verification_result.error}")

except Exception as e:
    print(f"[ERROR] Error during verification: {e}")
    raise

Step 4: Verifying authorization server token...
   Expected Audience: https://benefits.streamward.com

 Verifying authorization server token...
 Expected Issuer: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
 Expected Audience: https://benefits.streamward.com
 JWKS URI: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7/v1/keys
 Authorization server token verified successfully
 Subject: danny.fuhriman@okta.com
 Email: N/A
 Audience: https://benefits.streamward.com
 Issuer: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
 Scope: mcp:read
 Expires: 2026-02-03 06:30:08
[SUCCESS] Authorization server token is VALID!

Token Claims:
   Subject: danny.fuhriman@okta.com
   Email: N/A
   Issuer: https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7
   Audience: https://benefits.streamward.com
   Scope: mcp:read
   Expires At: 2026-02-03 06:30:08

Full Payload:
{
  "ver": 1,
  "jti": "AT.5EWm--aECM6kJ6VvDGc3gXJiK4NWpbBEK9F-PpTaJiY",
  "iss": "https://atko

---

## Resources

- [Okta AI SDK Documentation](https://github.com/indranilokg/okta-ai-sdk-python-proto)
- [ID-JAG - Identity Assertion Authorization Grant](https://datatracker.ietf.org/doc/draft-ietf-oauth-identity-assertion-authz-grant/)

---
